# 01 — Data Downloading

This notebook downloads project datasets from Cloudflare R2 (S3-compatible storage).

**Prerequisites:**
- AWS CLI installed (`brew install awscli`)
- R2 credentials configured in `scripts/r2.env` (see `scripts/r2.env.example`)

**Available datasets:**
| Version | Description | Source |
|---------|-------------|--------|
| v1 | Raw Hive-partitioned parquet (all VARCHAR columns) | R2 |
| v2 | Cleaned + typed parquet (66 countries, 25.1M rows) | R2 |
| v1_aux | Cultural distance matrices + country metadata | Generated locally |
| v3_features | Feature-engineered train/val/test splits | R2 |

**Run the cells for the version(s) you need.** Each section is independent.

In [1]:
from pathlib import Path
import os
import re
import subprocess
from typing import Dict, List, Tuple
import pandas as pd

# Path resolution — notebooks/ -> repo root
REPO_ROOT = Path.cwd().parent
SCRIPTS_DIR = REPO_ROOT / "scripts"
DATASETS_DIR = REPO_ROOT / "datasets"
R2_ENV_PATH = SCRIPTS_DIR / "r2.env"
R2_ENV_EXAMPLE_PATH = SCRIPTS_DIR / "r2.env.example"

print(f"Repo root:   {REPO_ROOT}")
print(f"Scripts dir: {SCRIPTS_DIR}")
print(f"Datasets dir: {DATASETS_DIR}")
print(f"R2 env file: {R2_ENV_PATH}")

Repo root:   /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB
Scripts dir: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/scripts
Datasets dir: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets
R2 env file: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/scripts/r2.env


In [2]:
REQUIRED_R2_KEYS = [
    "R2_ENDPOINT",
    "R2_BUCKET",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "DATASET_VERSION",
]


def _strip_balanced_quotes(value: str) -> str:
    if len(value) >= 2 and ((value[0] == '"' and value[-1] == '"') or (value[0] == "'" and value[-1] == "'")):
        return value[1:-1]
    return value


def parse_env_file(env_path: Path) -> Tuple[Dict[str, str], List[str]]:
    parsed: Dict[str, str] = {}
    errors: List[str] = []

    if not env_path.exists():
        errors.append(f"Missing env file: {env_path}")
        return parsed, errors

    for line_no, raw_line in enumerate(env_path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            errors.append(f"Line {line_no}: missing '=' separator")
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()

        if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", key):
            errors.append(f"Line {line_no}: invalid key '{key}'")
            continue

        dbl_quote_count = value.count('"')
        sgl_quote_count = value.count("'")
        if dbl_quote_count % 2 != 0 or sgl_quote_count % 2 != 0:
            errors.append(
                f"Line {line_no}: malformed quotes for key '{key}' -> {value!r}"
            )
            continue

        normalized = _strip_balanced_quotes(value).strip()

        if normalized.endswith('"') or normalized.endswith("'"):
            errors.append(
                f"Line {line_no}: value for '{key}' appears to have a trailing quote -> {value!r}"
            )

        parsed[key] = normalized

    missing = [k for k in REQUIRED_R2_KEYS if not parsed.get(k)]
    if missing:
        errors.append(f"Missing required keys: {missing}")

    return parsed, errors


r2_config, r2_config_errors = parse_env_file(R2_ENV_PATH)

if r2_config_errors:
    print("r2.env validation failed:")
    for err in r2_config_errors:
        print(f"  - {err}")
    if R2_ENV_EXAMPLE_PATH.exists():
        print(f"\nTemplate available at: {R2_ENV_EXAMPLE_PATH}")
else:
    print("r2.env validation passed")
    display(pd.DataFrame([r2_config]).T.rename(columns={0: "value"}))

r2.env validation passed


,value
R2_ENDPOINT,https://a96b93c5d97cddb48fc674255fb687c7.r2.cl...
R2_BUCKET,ml-group-ab-datasets
AWS_ACCESS_KEY_ID,fa162d7d8a7a374608e16c3858dfb6f6
AWS_SECRET_ACCESS_KEY,c810c10fdc699b06aa6d561f1d052387ec79771ea972db...
DATASET_VERSION,v2


In [3]:
def build_sanitized_env(base_env: Dict[str, str], overrides: Dict[str, str] | None = None) -> Dict[str, str]:
    if r2_config_errors:
        raise ValueError("Cannot build environment: r2.env is invalid.")

    env = dict(os.environ)
    for key in REQUIRED_R2_KEYS:
        env[key] = str(base_env[key]).strip()

    if overrides:
        for k, v in overrides.items():
            if v is not None:
                env[k] = str(v)

    return env


def run_bash_script(script_path: Path, env: Dict[str, str], cwd: Path, timeout: int = 3600) -> subprocess.CompletedProcess:
    if not script_path.exists():
        raise FileNotFoundError(f"Missing script: {script_path}")

    cmd = ["bash", str(script_path)]
    result = subprocess.run(
        cmd,
        cwd=str(cwd),
        env=env,
        capture_output=True,
        text=True,
        timeout=timeout,
    )

    print("--- stdout ---")
    print(result.stdout[-4000:] if result.stdout else "<empty>")
    if result.stderr:
        print("--- stderr ---")
        print(result.stderr[-4000:])
    print(f"Exit code: {result.returncode}")

    if result.returncode != 0:
        raise RuntimeError(
            "Script failed. Check stdout/stderr above. "
            "For permission-related preflight errors, try SKIP_R2_PREFLIGHT=1."
        )

    return result


def download_version(version: str) -> None:
    """Download a dataset version from R2."""
    download_root = DATASETS_DIR / version
    env = build_sanitized_env(r2_config, overrides={
        "DATASET_VERSION": version,
        "DOWNLOAD_ROOT": str(download_root),
    })
    run_bash_script(SCRIPTS_DIR / "download_from_r2.sh", env=env, cwd=REPO_ROOT)
    # Quick verification
    parquet_files = list(download_root.rglob("*.parquet"))
    print(f"\nVerification: {len(parquet_files)} parquet file(s) in {download_root}")


print("Shell helpers ready")

Shell helpers ready


## Download v1 (Raw Parquet)
Raw Spotify charts data, Hive-partitioned by year. All columns are VARCHAR — use DuckDB for queries.

In [4]:
download_version("v1")

--- stdout ---
Using config file: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/scripts/r2.env
Dataset version: v2
Download root: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/v1
R2 bucket: ml-group-ab-datasets
Running R2 preflight check ...
Download complete.
Local path: /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/v1/

Exit code: 0

Verification: 53 parquet file(s) in /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/v1


## Download v2 (Cleaned Parquet)
Cleaned dataset: proper types, 66 countries, 25.1M rows. Produced by `03_data_cleaning.ipynb`.

In [5]:
download_version("v2")

--- stdout ---
ile(s) remaining
Completed 797.5 MiB/810.9 MiB (58.3 MiB/s) with 1 file(s) remaining
Completed 797.8 MiB/810.9 MiB (58.2 MiB/s) with 1 file(s) remaining
Completed 798.0 MiB/810.9 MiB (58.2 MiB/s) with 1 file(s) remaining
Completed 798.3 MiB/810.9 MiB (58.2 MiB/s) with 1 file(s) remaining
Completed 798.5 MiB/810.9 MiB (58.2 MiB/s) with 1 file(s) remaining
Completed 798.8 MiB/810.9 MiB (58.2 MiB/s) with 1 file(s) remaining
Completed 799.0 MiB/810.9 MiB (58.2 MiB/s) with 1 file(s) remaining
Completed 799.3 MiB/810.9 MiB (58.2 MiB/s) with 1 file(s) remaining
Completed 799.5 MiB/810.9 MiB (58.2 MiB/s) with 1 file(s) remaining
Completed 799.8 MiB/810.9 MiB (58.1 MiB/s) with 1 file(s) remaining
Completed 800.0 MiB/810.9 MiB (58.1 MiB/s) with 1 file(s) remaining
Completed 800.3 MiB/810.9 MiB (58.1 MiB/s) with 1 file(s) remaining
Completed 800.5 MiB/810.9 MiB (58.1 MiB/s) with 1 file(s) remaining
Completed 800.8 MiB/810.9 MiB (58.1 MiB/s) with 1 file(s) remaining
Completed 801.0 

## Download v3_features (Train/Val/Test Splits)
Feature-engineered dataset with ~102 features per (track, target_country) pair. Produced by `05_feature_engineering_v2.ipynb`.

In [6]:
download_version("v3_features")

--- stdout ---
ompleted 797.4 MiB/810.9 MiB (58.0 MiB/s) with 1 file(s) remaining
Completed 797.6 MiB/810.9 MiB (58.0 MiB/s) with 1 file(s) remaining
Completed 797.9 MiB/810.9 MiB (58.0 MiB/s) with 1 file(s) remaining
Completed 798.1 MiB/810.9 MiB (57.9 MiB/s) with 1 file(s) remaining
Completed 798.4 MiB/810.9 MiB (57.9 MiB/s) with 1 file(s) remaining
Completed 798.6 MiB/810.9 MiB (57.9 MiB/s) with 1 file(s) remaining
Completed 798.9 MiB/810.9 MiB (57.8 MiB/s) with 1 file(s) remaining
Completed 799.1 MiB/810.9 MiB (57.9 MiB/s) with 1 file(s) remaining
Completed 799.4 MiB/810.9 MiB (57.9 MiB/s) with 1 file(s) remaining
Completed 799.6 MiB/810.9 MiB (57.8 MiB/s) with 1 file(s) remaining
Completed 799.9 MiB/810.9 MiB (57.8 MiB/s) with 1 file(s) remaining
Completed 800.1 MiB/810.9 MiB (57.8 MiB/s) with 1 file(s) remaining
Completed 800.4 MiB/810.9 MiB (57.7 MiB/s) with 1 file(s) remaining
Completed 800.6 MiB/810.9 MiB (57.7 MiB/s) with 1 file(s) remaining
Completed 800.9 MiB/810.9 MiB (57.

## Generate v1_aux (Auxiliary Tables)
Cultural distance matrices and country metadata. **Not on R2** — generated locally from CSVs in the repository.

In [7]:
aux_script = SCRIPTS_DIR / "prepare_auxiliary_datasets.sh"
result = subprocess.run(
    ["bash", str(aux_script)],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    timeout=120,
)
print("--- stdout ---")
print(result.stdout[-4000:] if result.stdout else "<empty>")
if result.stderr:
    print("--- stderr ---")
    print(result.stderr[-4000:])
print(f"Exit code: {result.returncode}")

aux_path = DATASETS_DIR / "v1_aux"
aux_files = list(aux_path.rglob("*.parquet"))
print(f"\nVerification: {len(aux_files)} parquet file(s) in {aux_path}")

--- stdout ---
Auxiliary dataset processing finished.
                 artifact  rows                                                                                                                           path
   cultural_distance_long 14042    /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/v1_aux/cultural_distance_long.parquet
   cultural_distance_top5   595    /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/v1_aux/cultural_distance_top5.parquet
countries_reference_clean   250 /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/v1_aux/countries_reference_clean.parquet

Exit code: 0

Verification: 3 parquet file(s) in /Users/leonschmidt/Projekte/Machine_Learning_Spotify/Git_Project/ML_Group_AB/datasets/v1_aux


## Verification
Quick check that downloaded datasets exist and contain parquet files.

In [8]:
for version in ["v1", "v2", "v3_features", "v1_aux"]:
    path = DATASETS_DIR / version
    if version in ("v1", "v2"):
        path = path / "full"
    exists = path.exists()
    n_files = len(list(path.rglob("*.parquet"))) if exists else 0
    status = f"{n_files} parquet files" if exists else "NOT FOUND"
    print(f"  {version:15s} {status}")

  v1              29 parquet files
  v2              5 parquet files
  v3_features     5 parquet files
  v1_aux          3 parquet files
